In [1]:
import json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from asforests.cb_computer import Callback

from experiments.problem_instance.problem_instance import ProblemInstance

from experiments.benchmark.benchmark import Benchmark
from experiments.benchmark.approaches import DatabaseWiseApproach
from experiments.benchmark._ground_truth_computer import GroundTruthComputer

import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from tqdm import tqdm

In [2]:
class XiTermPairCollector(Callback):
    
    def __init__(self):
        super().__init__()
        self.xi_pairs_list = []
       
    def on_xi_term_pair_computation(self, cov_updater):
        if cov_updater.name == "cond":
            self.xi_pairs_list.append(cov_updater.all_new_xi_pairs_available)

# Compute and Store Approach Errors on Different Datasets and Seeds

In [9]:
d_ensemble_sequence_seed = list(range(1))
max_budget = 64

for openmlid in [1049]:
    for data_seed in range(1):
        X, y = None, None
        for num_possible_ensemble_members in [max_budget]:
            for validation_size in [8, 64, 256, 1024]:
                pi = None
                for ensemble_sequence_seed in d_ensemble_sequence_seed:

                    print(f"{openmlid=}, {data_seed=}, {num_possible_ensemble_members=}, {validation_size=}")

                    result_folder = Path(f"xitermpairs/{openmlid}_{data_seed}_{num_possible_ensemble_members}_{validation_size}_{ensemble_sequence_seed}")
                    if not result_folder.exists():

                        # make dir
                        result_folder.mkdir(parents=True, exist_ok=True)
                            
                        # first load the data unless it has been loaded before for the suitable circumstances
                        if X is None:
                            X, y = fetch_openml(data_id=openmlid, return_X_y=True)
                            #X, _, y, _ = train_test_split(X, y, train_size=num_instances, random_state=0)
                            if isinstance(X, pd.DataFrame):
                                X = X.values
                            if isinstance(y, pd.Series):
                                y = y.values
                            
                        # create problem instance and compute ground truth
                        pi = ProblemInstance(
                            data_description=(X, y),
                            is_classification=True,
                            data_seed=data_seed,
                            ensemble_seed=0,
                            training_instances_per_class=0.5,
                            num_possible_ensemble_members=num_possible_ensemble_members,
                            validation_size=validation_size,
                            num_samples_allowed_for_ground_truth_approximation=10,
                            n_checkpoints=np.array([1, 2, 10, 100, 1000]),
                            t_checkpoints=np.array([1, 2, 10, 100, 1000])
                        )

                        # get approach that contains ground truth values for the right hand side of Theorem 1
                        #agt_iid = pi.approach_for_gt_iid_case
                        #agt_cond = pi.approach_for_gt_conditional_case

                        # 
                        cb = XiTermPairCollector()
                        a = DatabaseWiseApproach(
                            estimated_parameters="V[Z_nt|D_val]",
                            max_number_of_recent_members_to_combine_with=100,
                            callbacks=[cb]
                        )
                        a.reset()
                        a.tell_ground_truth_labels(pi.y_oh_val)

                        # compute error in estimation of conditional covariance terms
                        for pm in pi.get_prediction_matrix_generator(
                            ensemble_sequence_seed=ensemble_sequence_seed,
                            only_validation_data=True
                        ):
                            a.receive_deviations_of_new_ensemble_member(pm)
                            print(len(cb.xi_pairs_list), len(cb.xi_pairs_list[-1]))
                            cb.xi_pairs_list[-1].to_csv(f"{result_folder}/{len(cb.xi_pairs_list)}.csv", index=False)
                            if len(cb.xi_pairs_list) >= max_budget: # stop after 30 rounds
                                break

openmlid=1049, data_seed=0, num_possible_ensemble_members=64, validation_size=8
1 1
2 15
3 65
4 175
5 369
6 671
7 1105
8 1695
9 2465
10 3439
11 4641
12 6095
13 7825
14 9855
15 12209
16 14911
17 17985
18 21455
19 25345
20 29679
21 34481
22 39775
23 45585
24 51935
25 58849
26 66351
27 74465
28 83215
29 92625
30 102719
31 113521
32 125055
33 137345
34 150415
35 164289
36 178991
37 194545
38 210975
39 228305
40 246559
41 265761
42 285935
43 307105
44 329295
45 352529
46 376831
47 402225
48 428735
49 456385
50 485199
51 515201
52 546415
53 578865
54 612575
55 647569
56 683871
57 721505
58 760495
59 800865
60 842639
61 885841
62 930495
63 976625
64 1024255
openmlid=1049, data_seed=0, num_possible_ensemble_members=64, validation_size=64
1 1
2 15
3 65
4 175
5 369
6 671
7 1105
8 1695
9 2465
10 3439
11 4641
12 6095
13 7825
14 9855
15 12209
16 14911
17 17985
18 21455
19 25345
20 29679
21 34481
22 39775
23 45585
24 51935
25 58849
26 66351
27 74465
28 83215
29 92625
30 102719
31 113521
32 125055
33